# Data Validation Report

This notebook validates the cleaned dataset before machine learning model development.

The objective is to ensure that the dataset satisfies all quality requirements for training a reliable Traffic Congestion Prediction model.

Load the Clean Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../dataset/processed/traffic_cleaned.csv")

print(df.shape)
df.head()

(179649, 17)


,segment_id,lat,lon,hour,avg_speed_kmh,density_veh_per_km,incidents,congestion_index,Congestion Level,date,month,day,day_of_week,is_weekend,is_peak_hour,time_of_day,congestion_category
0,SEG-0162239,6.863982,13.298207,14,95.3,23.1,1,20.6,Low,2024-12-16,December,Monday,Monday,0,0,Afternoon,Low
1,SEG-0117236,6.969183,14.701078,15,92.3,36.0,0,23.1,Low,2025-03-01,March,Saturday,Saturday,1,0,Afternoon,Low
2,SEG-0024260,4.801158,2.687999,11,51.9,32.1,0,42.4,Moderate,2024-11-16,November,Saturday,Saturday,1,0,Morning,Moderate
3,SEG-0093490,4.871314,8.517009,11,54.6,38.8,1,9.0,Low,2025-06-11,June,Wednesday,Wednesday,0,0,Morning,Low
4,SEG-0162248,6.062270,10.121534,7,29.6,55.4,0,67.1,High,2024-12-28,December,Saturday,Saturday,1,1,Morning,High


Create Validation Functions

In [2]:
def validation_result(test_name, passed, details):
    return {
        "Validation Test": test_name,
        "Status": "PASS ✅" if passed else "FAIL ❌",
        "Details": details
    }

Store validation Result

In [3]:
validation_results = []

Validation 1 – Missing Values

In [4]:
missing = df.isnull().sum().sum()

validation_results.append(
    validation_result(
        "Missing Values",
        missing == 0,
        f"Missing Values Found: {missing}"
    )
)

Validation 2 – Duplicate Records

In [5]:
duplicates = df.duplicated().sum()

validation_results.append(
    validation_result(
        "Duplicate Records",
        duplicates == 0,
        f"Duplicate Records: {duplicates}"
    )
)

Validation 3 – Average Speed

Average speed should never be negative.

In [7]:
negative_speed = (df["avg_speed_kmh"] < 0).sum()

validation_results.append(
    validation_result(
        "Average Speed Validation",
        negative_speed == 0,
        f"Negative Speeds: {negative_speed}"
    )
)

Validation 4 – Congestion Index

In [8]:
invalid_congestion = df[
    (df["congestion_index"] < 0) |
    (df["congestion_index"] > 100)
].shape[0]

validation_results.append(
    validation_result(
        "Congestion Index Range",
        invalid_congestion == 0,
        f"Invalid Values: {invalid_congestion}"
    )
)

Validation 5 – Hour Range

Hours should be

In [9]:
invalid_hour = df[
    ~df["hour"].between(0,23)
].shape[0]

validation_results.append(
    validation_result(
        "Hour Validation",
        invalid_hour == 0,
        f"Invalid Hours: {invalid_hour}"
    )
)

Validation 6 – Weekend Flag

In [10]:
invalid_weekend = df[
    ~df["is_weekend"].isin([0,1])
].shape[0]

validation_results.append(
    validation_result(
        "Weekend Flag",
        invalid_weekend == 0,
        f"Invalid Values: {invalid_weekend}"
    )
)

Validation 7 – Peak Hour Flag

In [11]:
invalid_peak = df[
    ~df["is_peak_hour"].isin([0,1])
].shape[0]

validation_results.append(
    validation_result(
        "Peak Hour Flag",
        invalid_peak == 0,
        f"Invalid Values: {invalid_peak}"
    )
)

Validation 8 – Congestion Category

In [12]:
expected = [
    "Low",
    "Moderate",
    "High",
    "Severe"
]

invalid_category = df[
    ~df["congestion_category"].isin(expected)
].shape[0]

validation_results.append(
    validation_result(
        "Congestion Category",
        invalid_category == 0,
        f"Invalid Categories: {invalid_category}"
    )
)

Validation 9 – Speed Outliers

In [13]:
speed_outliers = df[
    df["avg_speed_kmh"] > 180
].shape[0]

validation_results.append(
    validation_result(
        "Speed Outliers",
        speed_outliers == 0,
        f"Speeds Above 180 km/h: {speed_outliers}"
    )
)

Validation 10 – Traffic Density

In [14]:
negative_density = df[
    df["density_veh_per_km"] < 0
].shape[0]

validation_results.append(
    validation_result(
        "Traffic Density",
        negative_density == 0,
        f"Negative Density Values: {negative_density}"
    )
)

Create Validation Report

In [15]:
validation_report = pd.DataFrame(validation_results)

validation_report

,Validation Test,Status,Details
0,Missing Values,PASS ✅,Missing Values Found: 0
1,Duplicate Records,PASS ✅,Duplicate Records: 0
2,Average Speed Validation,PASS ✅,Negative Speeds: 0
3,Average Speed Validation,PASS ✅,Negative Speeds: 0
4,Congestion Index Range,PASS ✅,Invalid Values: 0
5,Hour Validation,PASS ✅,Invalid Hours: 0
6,Weekend Flag,PASS ✅,Invalid Values: 0
7,Peak Hour Flag,PASS ✅,Invalid Values: 0
8,Congestion Category,PASS ✅,Invalid Categories: 0
9,Speed Outliers,PASS ✅,Speeds Above 180 km/h: 0


Overall Validation Status

In [16]:
if (validation_report["Status"] == "PASS ✅").all():
    print("🎉 Dataset Validation Successful!")
    print("The dataset is ready for Machine Learning.")
else:
    print("⚠ Dataset Validation Failed.")
    print("Please review the failed validation tests.")

🎉 Dataset Validation Successful!
The dataset is ready for Machine Learning.


Save Validation Report

In [17]:
validation_report.to_csv(
    "../../docs/data_validation_report.csv",
    index=False
)

print("Validation report saved successfully.")

Validation report saved successfully.
